# Yalda Embedding Training

Fine-tune **Qwen3-Embedding-0.6B** for Persian retrieval with query-to-document
InfoNCE and relational knowledge distillation from precomputed **F2LLM-v2-8B** embeddings.
The final paper recipe uses `KD_ALPHA = 0.50` and a global batch size of 512.

Run the cells in order on a TPU runtime with internet access. The reported setup uses
eight TPU v5e devices. Edit the configuration cell before training; export and upload
are separate steps. Model weights are published to the **Hugging Face Hub**.


## 1. Install dependencies

Keep the TPU-specific JAX installation and the pinned JAX/Flax versions below.
If the runtime requests a kernel restart after installation, restart it and continue
with the imports.


In [ ]:
!pip install -U "jax[tpu]==0.9.2" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
!pip install flax==0.12.6
!pip install safetensors
!pip install datasets transformers optax matplotlib huggingface_hub


In [ ]:
import json
import os
import queue
import threading
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import flax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from datasets import load_dataset
from flax import nnx
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from jax.sharding import NamedSharding, PartitionSpec as P
from safetensors.numpy import load_file, save_file
from transformers import AutoTokenizer

print(f"JAX: {jax.__version__} | Flax: {flax.__version__}")


## 2. Configure the experiment

`KD_ALPHA` controls `(1 - KD_ALPHA) * InfoNCE + KD_ALPHA * KL(teacher || student)`.
Use `0.0` for InfoNCE only, `0.25` for the paper's Ablation 4, `0.50` for its final
model, or `1.0` for pure distillation.

Set `HF_TOKEN` through the runtime's environment/secrets, or enter it below for a local
session. Leave the published notebook free of tokens. `PUSH_REPO_ID` is the destination
Hugging Face model repository; set `UPLOAD_TO_HUB = True` to enable the final upload cell.
The backbone architecture below is specifically Qwen3-Embedding-0.6B.


In [ ]:
# Authentication and publication.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
PUSH_REPO_ID = "Yaldat/Yalda-Embedding"
UPLOAD_TO_HUB = False

# Model, data, and objective.
BASE_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
DATASET_ID = "Yaldat/Persian-Pairwise-Embedded"
KD_ALPHA = 0.50
STUDENT_TEMPERATURE = 0.05
TEACHER_TEMPERATURE = 0.07
TEACHER_DIMENSIONS = 1024
QUERY_INSTRUCTION = (
    "Given a web search query, retrieve relevant passages that answer the query"
)

# Training settings.
EPOCHS = 1
BATCH_SIZE = 512
MAX_LENGTH = 512
PEAK_LEARNING_RATE = 2e-5
WARMUP_FRACTION = 0.10
MAX_GRAD_NORM = 1.0
MODEL_SEED = 0
SHUFFLE_SEED = None  # The paper's final run did not fix the data permutation seed.
PREFETCH_BATCHES = 10
LOG_INTERVAL = 10
OUTPUT_DIR = Path("yalda_embedding_export")

if not 0.0 <= KD_ALPHA <= 1.0:
    raise ValueError("KD_ALPHA must be between 0 and 1.")
if STUDENT_TEMPERATURE <= 0 or TEACHER_TEMPERATURE <= 0:
    raise ValueError("Temperatures must be positive.")
if EPOCHS < 1 or BATCH_SIZE < 2 or MAX_LENGTH < 1:
    raise ValueError("Use at least one epoch, two pairs per batch, and one token.")
if not 0.0 < WARMUP_FRACTION < 1.0:
    raise ValueError("WARMUP_FRACTION must be between 0 and 1, exclusive.")
if PREFETCH_BATCHES < 1 or LOG_INTERVAL < 1:
    raise ValueError("Prefetch and logging intervals must be positive.")
if TEACHER_DIMENSIONS != 1024:
    raise ValueError("This recipe uses the first 1,024 teacher dimensions.")

hub_token = HF_TOKEN or None


## 3. Create the TPU mesh

The single `data` axis shards both minibatches and large parameter matrices across
devices. The loss operates on the global batch, providing 511 in-batch negatives
per query when `BATCH_SIZE = 512`.


In [ ]:
devices = jax.devices()
if any(device.platform != "tpu" for device in devices):
    raise RuntimeError("Select a TPU runtime before running this training notebook.")

num_devices = len(devices)
if BATCH_SIZE % num_devices:
    raise ValueError("BATCH_SIZE must be divisible by the number of TPU devices.")

mesh = jax.make_mesh(
    (num_devices,),
    ("data",),
    axis_types=(jax.sharding.AxisType.Auto,),
)

def named_sharding(*axes):
    return NamedSharding(mesh, P(*axes))

print(f"TPU devices: {num_devices} | Pairs per device: {BATCH_SIZE // num_devices}")


## 4. Define the student encoder

This Flax NNX implementation retains the original 28-layer Qwen3 embedding
architecture: grouped-query causal attention, per-head Q/K normalization, RoPE,
SwiGLU, and last non-padding token pooling. Parameters and normalization use FP32;
projections use BF16. Transformer blocks use activation rematerialization.


In [ ]:
@dataclass
class Qwen3EmbeddingConfig:
    vocab_size: int = 151669
    hidden_size: int = 1024
    head_dim: int = 128
    intermediate_size: int = 3072
    num_hidden_layers: int = 28
    num_attention_heads: int = 16
    num_key_value_heads: int = 8
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1_000_000.0
    param_dtype: object = jnp.float32
    dtype: object = jnp.bfloat16


def partitioned_initializer(axes):
    return nnx.with_partitioning(nnx.initializers.lecun_normal(), axes)


def compute_rope_cos_sin(seq_len, head_dim, theta):
    inv_freq = 1.0 / (
        theta ** (jnp.arange(0, head_dim, 2, dtype=jnp.float32) / head_dim)
    )
    positions = jnp.arange(seq_len, dtype=jnp.float32)
    frequencies = jnp.outer(positions, inv_freq)
    angles = jnp.concatenate((frequencies, frequencies), axis=-1)
    return (
        jnp.cos(angles)[None, :, None, :],
        jnp.sin(angles)[None, :, None, :],
    )


def apply_rotary_emb(q, k, cos, sin):
    def rotate_half(x):
        first, second = jnp.split(x, 2, axis=-1)
        return jnp.concatenate((-second, first), axis=-1)

    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


In [ ]:
class SwiGLUMLP(nnx.Module):
    def __init__(self, config, rngs):
        kwargs = dict(
            use_bias=False,
            dtype=config.dtype,
            param_dtype=config.param_dtype,
            rngs=rngs,
        )
        self.gate_proj = nnx.Linear(
            config.hidden_size, config.intermediate_size,
            kernel_init=partitioned_initializer((None, "data")), **kwargs,
        )
        self.up_proj = nnx.Linear(
            config.hidden_size, config.intermediate_size,
            kernel_init=partitioned_initializer((None, "data")), **kwargs,
        )
        self.down_proj = nnx.Linear(
            config.intermediate_size, config.hidden_size,
            kernel_init=partitioned_initializer(("data", None)), **kwargs,
        )

    def __call__(self, x):
        return self.down_proj(nnx.silu(self.gate_proj(x)) * self.up_proj(x))


class GroupQueryAttention(nnx.Module):
    def __init__(self, config, rngs):
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_queries_per_kv = self.num_heads // self.num_kv_heads
        kwargs = dict(
            use_bias=False,
            dtype=config.dtype,
            param_dtype=config.param_dtype,
            rngs=rngs,
        )
        self.q_proj = nnx.Linear(
            config.hidden_size, self.num_heads * self.head_dim,
            kernel_init=partitioned_initializer((None, "data")), **kwargs,
        )
        self.k_proj = nnx.Linear(
            config.hidden_size, self.num_kv_heads * self.head_dim,
            kernel_init=partitioned_initializer((None, "data")), **kwargs,
        )
        self.v_proj = nnx.Linear(
            config.hidden_size, self.num_kv_heads * self.head_dim,
            kernel_init=partitioned_initializer((None, "data")), **kwargs,
        )
        self.o_proj = nnx.Linear(
            self.num_heads * self.head_dim, config.hidden_size,
            kernel_init=partitioned_initializer(("data", None)), **kwargs,
        )
        norm_kwargs = dict(
            epsilon=config.rms_norm_eps,
            dtype=jnp.float32,
            param_dtype=jnp.float32,
            rngs=rngs,
        )
        self.q_norm = nnx.RMSNorm(self.head_dim, **norm_kwargs)
        self.k_norm = nnx.RMSNorm(self.head_dim, **norm_kwargs)

    def __call__(self, x, cos, sin, mask):
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).reshape(batch_size, seq_len, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).reshape(batch_size, seq_len, self.num_kv_heads, self.head_dim)

        q = self.q_norm(q.astype(jnp.float32)).astype(x.dtype)
        k = self.k_norm(k.astype(jnp.float32)).astype(x.dtype)
        q, k = apply_rotary_emb(
            q.astype(jnp.float32), k.astype(jnp.float32), cos, sin,
        )
        q, k = q.astype(x.dtype), k.astype(x.dtype)
        k = jnp.repeat(k, self.num_queries_per_kv, axis=2)
        v = jnp.repeat(v, self.num_queries_per_kv, axis=2)
        q, k, v = (
            q.transpose(0, 2, 1, 3),
            k.transpose(0, 2, 1, 3),
            v.transpose(0, 2, 1, 3),
        )

        scale = 1.0 / jnp.sqrt(self.head_dim)
        scores = jnp.matmul(q, k.transpose(0, 1, 3, 2)) * scale
        if mask is not None:
            scores = scores + mask.astype(scores.dtype)
        probabilities = jax.nn.softmax(scores.astype(jnp.float32), axis=-1)
        context = jnp.matmul(probabilities.astype(x.dtype), v)
        context = context.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, -1)
        return self.o_proj(context)


class QwenBlock(nnx.Module):
    def __init__(self, config, rngs):
        norm_kwargs = dict(
            epsilon=config.rms_norm_eps,
            dtype=jnp.float32,
            param_dtype=jnp.float32,
            rngs=rngs,
        )
        self.input_layernorm = nnx.RMSNorm(config.hidden_size, **norm_kwargs)
        self.self_attn = GroupQueryAttention(config, rngs)
        self.post_attention_layernorm = nnx.RMSNorm(config.hidden_size, **norm_kwargs)
        self.mlp = SwiGLUMLP(config, rngs)

    @nnx.remat
    def __call__(self, x, cos, sin, mask):
        normed = self.input_layernorm(x.astype(jnp.float32)).astype(x.dtype)
        x = x + self.self_attn(normed, cos, sin, mask)
        normed = self.post_attention_layernorm(x.astype(jnp.float32)).astype(x.dtype)
        return x + self.mlp(normed)


class Qwen3Embedding(nnx.Module):
    def __init__(self, config, rngs):
        self.config = config
        self.embed_tokens = nnx.Embed(
            config.vocab_size, config.hidden_size,
            param_dtype=config.param_dtype,
            dtype=config.dtype,
            embedding_init=partitioned_initializer((None, "data")),
            rngs=rngs,
        )
        self.layers = nnx.List(
            QwenBlock(config, rngs) for _ in range(config.num_hidden_layers)
        )
        self.norm = nnx.RMSNorm(
            config.hidden_size,
            epsilon=config.rms_norm_eps,
            dtype=jnp.float32,
            param_dtype=jnp.float32,
            rngs=rngs,
        )

    def __call__(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        x = self.embed_tokens(input_ids)
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len)))
        if attention_mask is None:
            mask = jnp.where(causal_mask == 0, -1e9, 0.0)[None, None, :, :]
        else:
            valid_keys = attention_mask[:, None, None, :]
            mask = jnp.where(causal_mask * valid_keys == 0, -1e9, 0.0)

        cos, sin = compute_rope_cos_sin(
            seq_len, self.config.head_dim, self.config.rope_theta,
        )
        for layer in self.layers:
            x = layer(x, cos, sin, mask)
        x = self.norm(x)

        if attention_mask is None:
            return x[:, -1, :]
        is_left_padding = jnp.sum(attention_mask[:, -1]) == batch_size
        last_indices = jnp.where(
            is_left_padding, seq_len - 1, jnp.sum(attention_mask, axis=1) - 1,
        )
        return x[jnp.arange(batch_size), last_indices, :]


## 5. Load the tokenizer and teacher-embedded dataset

Queries receive the fixed retrieval instruction; documents do not. Each epoch shuffles
the training split once and drops its incomplete final batch. Teacher vectors are
truncated to their first 1,024 Matryoshka dimensions and normalized inside the loss.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID, padding_side="left", token=hub_token,
)
training_data = load_dataset(DATASET_ID, split="train", token=hub_token)
steps_per_epoch = len(training_data) // BATCH_SIZE
num_steps = steps_per_epoch * EPOCHS
warmup_steps = int(num_steps * WARMUP_FRACTION)
if steps_per_epoch < 1 or warmup_steps < 1:
    raise ValueError("The dataset is too small for the configured batch/warmup settings.")

print(f"Training pairs: {len(training_data):,}")
print(f"Steps: {num_steps:,} | Warmup steps: {warmup_steps:,}")
print(f"Dropped pairs per epoch: {len(training_data) % BATCH_SIZE}")


def format_query(query):
    return f"Instruct: {QUERY_INSTRUCTION}\nQuery:{query}"


def get_batches(data):
    rng = np.random.default_rng(SHUFFLE_SEED)
    for _ in range(EPOCHS):
        indices = rng.permutation(len(data))
        for start in range(0, len(data) - BATCH_SIZE + 1, BATCH_SIZE):
            batch = data[indices[start:start + BATCH_SIZE].tolist()]
            tokenization = dict(
                padding="max_length", truncation=True,
                max_length=MAX_LENGTH, return_tensors="np",
            )
            queries = tokenizer(
                [format_query(query) for query in batch["question"]], **tokenization,
            )
            documents = tokenizer(batch["context"], **tokenization)
            teacher_queries = np.asarray(batch["question_embedded"], dtype=np.float32)
            teacher_documents = np.asarray(batch["context_embedded"], dtype=np.float32)
            if min(teacher_queries.shape[1], teacher_documents.shape[1]) < TEACHER_DIMENSIONS:
                raise ValueError("Teacher embeddings have fewer than 1,024 dimensions.")
            yield (
                queries["input_ids"], documents["input_ids"],
                queries["attention_mask"], documents["attention_mask"],
                teacher_queries[:, :TEACHER_DIMENSIONS],
                teacher_documents[:, :TEACHER_DIMENSIONS],
            )


def prefetch_generator(generator, max_prefetch):
    pending = queue.Queue(maxsize=max_prefetch)
    sentinel = object()

    def worker():
        try:
            for batch in generator:
                pending.put(batch)
        except Exception as error:
            pending.put(error)
        finally:
            pending.put(sentinel)

    threading.Thread(target=worker, daemon=True).start()
    while True:
        item = pending.get()
        if item is sentinel:
            return
        if isinstance(item, Exception):
            raise item
        yield item


## 6. Define InfoNCE and relational distillation

Both objectives operate on the full query-document similarity matrix. Distillation
minimizes forward KL divergence between teacher and student row-wise distributions;
it does not match their vector coordinates. Similarities and losses use FP32.


In [ ]:
def l2_normalize(embeddings, eps=1e-6):
    embeddings = embeddings.astype(jnp.float32)

    norms = jnp.linalg.norm(
        embeddings,
        axis=-1,
        keepdims=True,
    )

    return embeddings / jnp.maximum(norms, eps)


def infonce_loss(
    q_embeddings,
    k_embeddings,
    teacher_q_embeddings,
    teacher_k_embeddings,
    temperature=0.05,
    teacher_temperature=0.07,
    kd_alpha=0.50,
):
    """
    Combines hard-label InfoNCE with relational knowledge distillation.

    The teacher and student embeddings are independently L2-normalized.
    Knowledge distillation matches their query-document similarity
    distributions instead of directly matching vector coordinates.
    """

    # Student embeddings.
    q_emb = l2_normalize(q_embeddings)
    k_emb = l2_normalize(k_embeddings)

    # Teacher embeddings. This also renormalizes the truncated
    # 1024-dimensional Matryoshka prefixes.
    teacher_q_emb = l2_normalize(
        jax.lax.stop_gradient(teacher_q_embeddings)
    )
    teacher_k_emb = l2_normalize(
        jax.lax.stop_gradient(teacher_k_embeddings)
    )

    # Student query-document similarity matrix.
    student_logits = (
        jnp.matmul(q_emb, k_emb.T) / temperature
    )

    # Teacher query-document similarity matrix.
    teacher_logits = (
        jnp.matmul(teacher_q_emb, teacher_k_emb.T)
        / teacher_temperature
    )

    teacher_logits = jax.lax.stop_gradient(teacher_logits)

    # Standard hard-label InfoNCE.
    batch_size = q_emb.shape[0]
    labels = jnp.arange(batch_size)

    hard_infonce_loss = (
        optax.softmax_cross_entropy_with_integer_labels(
            logits=student_logits,
            labels=labels,
        ).mean()
    )

    # Soft teacher relevance distribution.
    teacher_probabilities = jax.lax.stop_gradient(
        jax.nn.softmax(teacher_logits, axis=-1)
    )

    teacher_log_probabilities = jax.lax.stop_gradient(
        jax.nn.log_softmax(teacher_logits, axis=-1)
    )

    student_log_probabilities = jax.nn.log_softmax(
        student_logits,
        axis=-1,
    )

    # KL(teacher || student).
    knowledge_distillation_loss = jnp.sum(
        teacher_probabilities
        * (
            teacher_log_probabilities
            - student_log_probabilities
        ),
        axis=-1,
    ).mean()

    total_loss = (
        (1.0 - kd_alpha) * hard_infonce_loss
        + kd_alpha * knowledge_distillation_loss
    )

    return (
        total_loss,
        hard_infonce_loss,
        knowledge_distillation_loss,
    )


@nnx.jit
def train_step(
    model: nnx.Module,
    optimizer: nnx.Optimizer,
    queries,
    keys,
    teacher_queries,
    teacher_keys,
    query_masks=None,
    key_masks=None,
    temperature=0.05,
    teacher_temperature=0.07,
    kd_alpha=0.50,
):
    def loss_fn(model):
        # Student forward passes.
        q_emb = model(queries, query_masks)
        k_emb = model(keys, key_masks)

        # Perform all similarity and loss calculations in FP32.
        q_emb = q_emb.astype(jnp.float32)
        k_emb = k_emb.astype(jnp.float32)

        teacher_q_emb = teacher_queries.astype(jnp.float32)
        teacher_k_emb = teacher_keys.astype(jnp.float32)

        (
            total_loss,
            hard_infonce_loss,
            knowledge_distillation_loss,
        ) = infonce_loss(
            q_embeddings=q_emb,
            k_embeddings=k_emb,
            teacher_q_embeddings=teacher_q_emb,
            teacher_k_embeddings=teacher_k_emb,
            temperature=temperature,
            teacher_temperature=teacher_temperature,
            kd_alpha=kd_alpha,
        )

        return total_loss, (
            hard_infonce_loss,
            knowledge_distillation_loss,
        )

    (
        loss,
        (
            hard_infonce_loss,
            knowledge_distillation_loss,
        ),
    ), grads = nnx.value_and_grad(
        loss_fn,
        has_aux=True,
    )(model)

    optimizer.update(model, grads)

    return (
        loss,
        hard_infonce_loss,
        knowledge_distillation_loss,
    )


## 7. Initialize the pretrained model and optimizer

Hugging Face linear weights are transposed into Flax's kernel layout. Pretrained
weights retain each parameter's device sharding. AdamW uses global-norm clipping,
linear warmup, and cosine decay.


In [ ]:
def update_weights_inplace(weights, model):
    state = nnx.state(model)

    def load_parameter(weight_name, reference, transpose=False):
        array = weights[weight_name].T if transpose else weights[weight_name]
        array = jnp.asarray(array, dtype=jnp.float32)
        return nnx.Param(jax.device_put(array, reference.sharding))

    state.norm.scale = load_parameter("norm.weight", state.norm.scale)
    state.embed_tokens.embedding = load_parameter(
        "embed_tokens.weight", state.embed_tokens.embedding,
    )
    for index in range(len(state.layers)):
        layer = state.layers[index]
        prefix = f"layers.{index}"
        for name in ("input_layernorm", "post_attention_layernorm"):
            module = getattr(layer, name)
            module.scale = load_parameter(f"{prefix}.{name}.weight", module.scale)
        for name in ("q_norm", "k_norm"):
            module = getattr(layer.self_attn, name)
            module.scale = load_parameter(
                f"{prefix}.self_attn.{name}.weight", module.scale,
            )
        for group, names in (
            ("mlp", ("gate_proj", "up_proj", "down_proj")),
            ("self_attn", ("q_proj", "k_proj", "v_proj", "o_proj")),
        ):
            for name in names:
                module = getattr(getattr(layer, group), name)
                module.kernel = load_parameter(
                    f"{prefix}.{group}.{name}.weight", module.kernel, transpose=True,
                )
    nnx.update(model, state)


weights_path = hf_hub_download(
    repo_id=BASE_MODEL_ID, filename="model.safetensors", token=hub_token,
)
with jax.set_mesh(mesh):
    config = Qwen3EmbeddingConfig()
    model = Qwen3Embedding(config, rngs=nnx.Rngs(params=MODEL_SEED))
    pretrained_weights = load_file(weights_path)
    update_weights_inplace(pretrained_weights, model)
    del pretrained_weights

    learning_rate_schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=PEAK_LEARNING_RATE,
        warmup_steps=warmup_steps,
        decay_steps=num_steps,
        end_value=0.0,
    )
    optimizer = nnx.Optimizer(
        model,
        optax.chain(
            optax.clip_by_global_norm(MAX_GRAD_NORM),
            optax.adamw(learning_rate=learning_rate_schedule),
        ),
        wrt=nnx.Param,
    )


## 8. Train and save the loss history

Log interval means for the combined loss, InfoNCE, and KD. The saved configuration
excludes the authentication token. With the paper's dataset size, one epoch consumes
1,035,776 pairs in 2,023 updates and omits 205 pairs.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_config = {
    "base_model": BASE_MODEL_ID,
    "dataset": DATASET_ID,
    "push_repo_id": PUSH_REPO_ID,
    "kd_alpha": KD_ALPHA,
    "student_temperature": STUDENT_TEMPERATURE,
    "teacher_temperature": TEACHER_TEMPERATURE,
    "teacher_dimensions": TEACHER_DIMENSIONS,
    "query_instruction": QUERY_INSTRUCTION,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "peak_learning_rate": PEAK_LEARNING_RATE,
    "warmup_steps": warmup_steps,
    "max_grad_norm": MAX_GRAD_NORM,
    "model_seed": MODEL_SEED,
    "shuffle_seed": SHUFFLE_SEED,
    "training_pairs": len(training_data),
    "optimization_steps": num_steps,
    "tpu_devices": num_devices,
    "jax_version": jax.__version__,
    "flax_version": flax.__version__,
}
(OUTPUT_DIR / "training_config.json").write_text(
    json.dumps(run_config, indent=2) + "\n", encoding="utf-8",
)

batches = prefetch_generator(get_batches(training_data), PREFETCH_BATCHES)
history = []
interval_losses = np.zeros(3, dtype=np.float64)
interval_count = 0
interval_start = perf_counter()
print(f"Training for {num_steps:,} steps with KD_ALPHA={KD_ALPHA}.")

with jax.set_mesh(mesh):
    for step, batch in enumerate(batches, start=1):
        queries, documents, query_masks, document_masks, teacher_queries, teacher_documents = (
            jax.device_put(array, named_sharding("data")) for array in batch
        )
        losses = train_step(
            model, optimizer, queries, documents, teacher_queries, teacher_documents,
            query_masks, document_masks,
            temperature=STUDENT_TEMPERATURE,
            teacher_temperature=TEACHER_TEMPERATURE,
            kd_alpha=KD_ALPHA,
        )
        interval_losses += np.array([float(value) for value in losses])
        interval_count += 1

        if step % LOG_INTERVAL == 0 or step == num_steps:
            mean_losses = interval_losses / interval_count
            seconds_per_step = (perf_counter() - interval_start) / interval_count
            record = {
                "step": step,
                "total_loss": float(mean_losses[0]),
                "infonce_loss": float(mean_losses[1]),
                "kd_loss": float(mean_losses[2]),
                "seconds_per_step": seconds_per_step,
            }
            history.append(record)
            (OUTPUT_DIR / "training_history.json").write_text(
                json.dumps(history, indent=2) + "\n", encoding="utf-8",
            )
            print(
                f"Step {step}/{num_steps} | Total: {mean_losses[0]:.6f} | "
                f"InfoNCE: {mean_losses[1]:.6f} | KD: {mean_losses[2]:.6f} | "
                f"Seconds/step: {seconds_per_step:.3f}"
            )
            interval_losses.fill(0)
            interval_count = 0
            interval_start = perf_counter()

print("Training complete.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for field, label in (
    ("total_loss", "Combined loss"),
    ("infonce_loss", "InfoNCE"),
    ("kd_loss", "Knowledge distillation"),
):
    ax.plot([row["step"] for row in history], [row[field] for row in history], label=label)
ax.set(title=f"Yalda Embedding training (KD alpha = {KD_ALPHA})", xlabel="Step", ylabel="Loss")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_loss_curve.png", dpi=300)
plt.show()


## 9. Export to Hugging Face format

Export FP32 Safetensors and inherit the base model's configuration, tokenizer, and
Sentence Transformers pooling files. Linear kernels are transposed back to PyTorch
layout. The export directory also contains this run's settings, loss history, and plot.


In [ ]:
def export_model(model, output_dir):
    snapshot_download(
        repo_id=BASE_MODEL_ID,
        local_dir=str(output_dir),
        token=hub_token,
        ignore_patterns=["*.bin", "*.h5", "*.msgpack", "*.safetensors", "README.md"],
    )
    state = nnx.state(model)

    def as_array(parameter):
        value = parameter.value if hasattr(parameter, "value") else parameter
        return np.asarray(value, dtype=np.float32)

    weights = {
        "norm.weight": as_array(state.norm.scale),
        "embed_tokens.weight": as_array(state.embed_tokens.embedding),
    }
    for index in range(len(state.layers)):
        layer = state.layers[index]
        prefix = f"layers.{index}"
        for name in ("input_layernorm", "post_attention_layernorm"):
            weights[f"{prefix}.{name}.weight"] = as_array(getattr(layer, name).scale)
        for name in ("q_norm", "k_norm"):
            weights[f"{prefix}.self_attn.{name}.weight"] = as_array(
                getattr(layer.self_attn, name).scale,
            )
        for group, names in (
            ("mlp", ("gate_proj", "up_proj", "down_proj")),
            ("self_attn", ("q_proj", "k_proj", "v_proj", "o_proj")),
        ):
            for name in names:
                module = getattr(getattr(layer, group), name)
                weights[f"{prefix}.{group}.{name}.weight"] = as_array(module.kernel).T
    save_file(weights, str(output_dir / "model.safetensors"), metadata={"format": "pt"})
    print(f"Model exported to {output_dir.resolve()}")


export_model(model, OUTPUT_DIR)


## 10. Upload the exported model

This cell publishes to `PUSH_REPO_ID` when `UPLOAD_TO_HUB` is enabled. It uses the token
from the configuration cell and never prints it.


In [ ]:
if UPLOAD_TO_HUB:
    if not HF_TOKEN:
        raise ValueError("Set HF_TOKEN before uploading the model.")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=PUSH_REPO_ID, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=PUSH_REPO_ID,
        repo_type="model",
        ignore_patterns=[".cache/**"],
        commit_message=f"Upload Yalda Embedding training run (KD alpha={KD_ALPHA})",
    )
    print(f"Published model: https://huggingface.co/{PUSH_REPO_ID}")
else:
    print("Export ready. Set UPLOAD_TO_HUB=True to publish it to the Hugging Face Hub.")
